# 2. Model Training

This notebook trains the 5-tier HP prediction models using engineered features.

**Input:** `helper_files/engineered_features.parquet`  
**Output:** `pickled_models/hp_model_cr*.pkl`

In [ ]:
import pandas as pd
import numpy as np
import sys
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, '.')

from helper_files import (
    get_phase3_features,
    train_constrained_model,
    ConstrainedModel,
    calculate_r2,
    calculate_mae,
    save_model,
    summarize_model_performance,
    extract_family,
)

print("Imports successful")

## Load Engineered Features

In [ ]:
# Load engineered features
df = pd.read_parquet('helper_files/engineered_features.parquet')
print(f"Loaded {len(df)} monsters with {len(df.columns)} features")

In [ ]:
# Get Phase 3 features
phase3_features = get_phase3_features()
print(f"Phase 3 features: {len(phase3_features)}")

## Split by CR Tier

In [ ]:
# Split by CR tier
df_cr1 = df[df['cr_tier'] == 'cr1'].copy()
df_cr2 = df[df['cr_tier'] == 'cr2'].copy()
df_cr3 = df[df['cr_tier'] == 'cr3'].copy()
df_cr4 = df[df['cr_tier'] == 'cr4'].copy()
df_cr5 = df[df['cr_tier'] == 'cr5'].copy()

print(f"CR < 1:    {len(df_cr1)} monsters")
print(f"CR 1-4:    {len(df_cr2)} monsters")
print(f"CR 5-10:   {len(df_cr3)} monsters")
print(f"CR 11-16:  {len(df_cr4)} monsters")
print(f"CR > 16:   {len(df_cr5)} monsters")

## Train/Test Split Strategy

Using family-based splitting to avoid data leakage.

In [ ]:
# Manual train/test split by creature family
# Simple creatures -> all training
# Complex creatures -> split evenly

def manual_train_test_split(df_tier, test_ratio=0.2, random_state=42):
    """Split data using family-based strategy."""
    np.random.seed(random_state)
    
    # Simple families (all training)
    simple_families = ['beast', 'humanoid', 'giant']
    
    # Creatures from simple families
    simple_mask = df_tier['family'].isin(simple_families)
    
    # Get unique complex families
    complex_families = df_tier[~simple_mask]['family'].unique()
    
    # Shuffle and split complex families
    np.random.shuffle(complex_families)
    n_test = max(1, int(len(complex_families) * test_ratio))
    test_families = set(complex_families[:n_test])
    
    # Create masks
    train_mask = simple_mask | ~df_tier['family'].isin(test_families)
    test_mask = ~train_mask
    
    return df_tier[train_mask], df_tier[test_mask]

# For small tiers (CR 11-16, CR > 16), use all data for training
def split_by_tier(df_tier, tier_name):
    if len(df_tier) < 30:
        print(f"  {tier_name}: Using all {len(df_tier)} samples for training (small tier)")
        return df_tier, df_tier  # Train and test on same data for small tiers
    else:
        train, test = manual_train_test_split(df_tier)
        print(f"  {tier_name}: {len(train)} train, {len(test)} test")
        return train, test

In [ ]:
# Split each tier
print("Splitting data:")
train_cr1, test_cr1 = split_by_tier(df_cr1, 'CR < 1')
train_cr2, test_cr2 = split_by_tier(df_cr2, 'CR 1-4')
train_cr3, test_cr3 = split_by_tier(df_cr3, 'CR 5-10')
train_cr4, test_cr4 = split_by_tier(df_cr4, 'CR 11-16')
train_cr5, test_cr5 = split_by_tier(df_cr5, 'CR > 16')

## Train Models

In [ ]:
def train_tier_model(train_df, test_df, tier_name, phase3_features):
    """Train a model for a single CR tier."""
    print(f"\n{'='*60}")
    print(f"Training {tier_name} model...")
    print(f"{'='*60}")
    
    # Prepare features
    X_train = train_df[phase3_features].fillna(0).values
    y_train = train_df['residual_hp'].values
    
    X_test = test_df[phase3_features].fillna(0).values
    y_test = test_df['residual_hp'].values
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train constrained model
    coefficients, intercept = train_constrained_model(
        X_train_scaled, y_train, phase3_features, scaler
    )
    
    # Create model object
    model = ConstrainedModel(coefficients, intercept)
    
    # Evaluate on test set
    y_pred_residual = model.predict(X_test_scaled)
    
    # Calculate full HP predictions
    y_pred_hp = test_df['hp_after_phase2'].values + y_pred_residual
    y_actual_hp = test_df['actual_hp'].values
    
    # Calculate metrics
    r2 = calculate_r2(y_actual_hp, y_pred_hp)
    mae = calculate_mae(y_actual_hp, y_pred_hp)
    
    print(f"\n{tier_name} Results:")
    print(f"   Training samples: {len(y_train)}")
    print(f"   Test R²:  {r2:.4f}")
    print(f"   Test MAE: {mae:.2f} HP")
    
    return {
        'model': model,
        'scaler': scaler,
        'train_count': len(y_train),
        'test_r2': r2,
        'test_mae': mae,
    }

In [ ]:
# Train all models
results = {}

results['cr1'] = train_tier_model(train_cr1, test_cr1, 'CR < 1', phase3_features)
results['cr2'] = train_tier_model(train_cr2, test_cr2, 'CR 1-4', phase3_features)
results['cr3'] = train_tier_model(train_cr3, test_cr3, 'CR 5-10', phase3_features)
results['cr4'] = train_tier_model(train_cr4, test_cr4, 'CR 11-16', phase3_features)
results['cr5'] = train_tier_model(train_cr5, test_cr5, 'CR > 16', phase3_features)

In [ ]:
# Summary
summarize_model_performance(results)

## Save Models

In [ ]:
import os

# Ensure output directory exists
os.makedirs('../pickled_models', exist_ok=True)

# Save each model
for tier in ['cr1', 'cr2', 'cr3', 'cr4', 'cr5']:
    filepath = f'../pickled_models/hp_model_{tier}.pkl'
    save_model(
        results[tier]['model'],
        results[tier]['scaler'],
        phase3_features,
        filepath
    )
    print(f"Saved {tier} model to {filepath}")

print("\nAll models saved successfully!")

In [ ]:
# Display top feature coefficients for each tier
for tier in ['cr1', 'cr2', 'cr3', 'cr4', 'cr5']:
    print(f"\n{tier.upper()} Top Features by Coefficient:")
    coefs = results[tier]['model'].coef_
    coef_df = pd.DataFrame({
        'feature': phase3_features,
        'coefficient': coefs
    }).sort_values('coefficient', key=abs, ascending=False)
    
    print(coef_df.head(10).to_string(index=False))